<a href="https://colab.research.google.com/github/mariliazago0/fundamentos01/blob/main/notebooks/encontro01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install tinydb

In [28]:
# programas/bibliotecas utilizados no script/codigo
import httpx # Responsável pelas requisições web
from bs4 import BeautifulSoup # Responsável por realizar o web scraping (coletar os dados)
from tinydb import TinyDB, Query

In [29]:
def inserir_no_banco(dados, link_noticia):
  arquivo_banco_dados = "nota_mre.json"
  db = TinyDB(arquivo_banco_dados)


  # Evitar dados repetidos no banco
  Buscar = Query()
  verificar_link = db.contains(Buscar.link == link_noticia)

  if not verificar_link:
    print("Inserindo nova informação no banco")
    db.insert(dados)
  else:
    print("Link já existe no banco. Esta informação não será inserida novamente")

In [30]:
# Variável e tipos de dados (string, lista, numero)
paginas = ["https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0"]

def acessa_pagina (link):
  print (f"Estamos na pagina:{link}")

  # Define headers para a requisição, simulando um navegador
  headers = {
      'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
      'Accept-Language': 'en-US,en;q=0.9',
      'Accept-Encoding': 'gzip, deflate, br',
      'Connection': 'keep-alive',
  }

  timeout = httpx.Timeout(connect=20.0, read=30.0, write=20.0, pool=10.0)
  pag_web = httpx.get(link, headers=headers, timeout=timeout)
  bs = BeautifulSoup(pag_web, "html.parser")
  return bs

# loop for
# beautifulsoap >> find e find_all

for pagina in paginas:
  pagina_inteira = acessa_pagina(pagina)
  lista_noticias = pagina_inteira.find("div", attrs={"id": "content-core"}).find_all("article")
  for noticia in lista_noticias:
    # titulo
    try:
      titulo = noticia.find("h2", attrs={"class": "tileHeadline"}).text.strip()
      print(titulo)
    except:
        titulo = ""

    #link
    try:
      link_noticia = noticia.a["href"]
      print(link_noticia)
    except:
      link_noticia = ""
    # numero da nota - exemplo: NOTA À IMPRENSA Nº 72
    # numero da nota - exemplo: NOTA À IMPRENSA N° 590
    num_nota = noticia.find("span", attrs={"class": "subtitle"}).text.strip()
    # print(num_nota)
    # num_nota = noticia.find(attrs={"class": "subtitle"}).text.strip()
    num_nota = num_nota.replace("NOTA À IMPRENSA N°", "").replace("NOTA À IMPRENSA Nº", "").strip()
    print(num_nota)
    print("###")
    # data
    # horário
    data_hora = noticia.find_all("span",attrs={"class": "summary-view-icon"})
    data= data_hora[0].text.strip()
    hora = data_hora[1].text.strip()
    print(data)
    print(hora)
    conteudo = acessa_pagina (link_noticia)
    paragrafos = conteudo.find("div", attrs={"property":"rnews:articleBody"}).find_all("p")
    lista_paragrafos = []
    for paragrafo in paragrafos:
      lista_paragrafos.append(paragrafo.text.strip())
    print(lista_paragrafos)
    # função para inserir dados coletados no banco
    dados = {
        "titulo": titulo,
        "link": link_noticia,
        "data": data,
        "hora": hora,
        "num_nota": num_nota,
        "paragrafo": lista_paragrafos
    }
    inserir_no_banco(dados,link_noticia)

Estamos na pagina:https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0
Abertura de mercado para o Brasil no Chile - Nota Conjunta MRE/MAPA
https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/abertura-de-mercado-para-o-brasil-no-chile-nota-conjunta-mre-mapa
150
###
29/04/2026
17h49
Estamos na pagina:https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/abertura-de-mercado-para-o-brasil-no-chile-nota-conjunta-mre-mapa
['O governo brasileiro concluiu negociações que permitirão ao Brasil exportar grãos secos de destilaria (DDG/DDGS) para o Chile.', 'A abertura beneficia importante insumo agrícola gerado a partir da produção de etanol de milho. Os grãos secos de destilaria (DDG/DDGS) são utilizados na produção de ração para aves, bovinos e suínos.', 'Em 2025, o Brasil exportou mais de US$ 2,2 bilhões em produtos agropecuários para o Chile, com destaque para carnes, produtos florestais e o complexo 

In [31]:
import pandas as pd
import json

## Abrindo o rquivo json
with open("nota_mre.json") as f:
  raw = json.load(f)

df = pd.DataFrame.from_dict(raw["_default"], orient="index")

df

,titulo,link,data,hora,num_nota,paragrafo
1,Abertura de mercado para o Brasil no Chile - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,29/04/2026,17h49,150,[O governo brasileiro concluiu negociações que...
2,Promulgação do Acordo Provisório de Comércio e...,https://www.gov.br/mre/pt-br/canais_atendiment...,28/04/2026,19h20,149,[O Presidente Luiz Inácio Lula da Silva assino...
3,Mortes de brasileiros no Líbano em decorrência...,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,19h04,148,"[O governo brasileiro tomou conhecimento, com ..."
4,Ataques terroristas no Mali,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,13h38,147,[O Governo brasileiro manifesta grave preocupa...
5,Atentado na Colômbia,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,08h45,146,[O governo brasileiro condena o ataque perpetr...
6,Apresentação da candidatura do Professor Georg...,https://www.gov.br/mre/pt-br/canais_atendiment...,24/04/2026,15h57,145,"[Realizou-se, em 23 de abril, coquetel de apre..."
7,"Aberturas de mercado para o Brasil em Cuba, na...",https://www.gov.br/mre/pt-br/canais_atendiment...,24/04/2026,15h48,144,[O governo brasileiro concluiu negociações que...
8,Aberturas de mercado para o Brasil no Togo - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,22/04/2026,19h20,143,[O governo brasileiro concluiu negociações que...
9,Atos adotados por ocasião da Visita de Estado ...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,13h13,142,[Foram adotados os seguintes atos por ocasião ...
10,Declaração Conjunta Brasil-Alemanha sobre apoi...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,11h12,141,[No contexto da reunião bilateral entre o Pres...


In [32]:
# saber quantidade de linhas e colunas do dataframe
df.shape

(30, 6)

In [33]:
# saber colunas disponiveis
df.columns

Index(['titulo', 'link', 'data', 'hora', 'num_nota', 'paragrafo'], dtype='object')

In [34]:
# selecionar uma coluna em especifico
df["titulo"]

,titulo
1,Abertura de mercado para o Brasil no Chile - N...
2,Promulgação do Acordo Provisório de Comércio e...
3,Mortes de brasileiros no Líbano em decorrência...
4,Ataques terroristas no Mali
5,Atentado na Colômbia
6,Apresentação da candidatura do Professor Georg...
7,"Aberturas de mercado para o Brasil em Cuba, na..."
8,Aberturas de mercado para o Brasil no Togo - N...
9,Atos adotados por ocasião da Visita de Estado ...
10,Declaração Conjunta Brasil-Alemanha sobre apoi...


In [35]:

# delimitar colunas do dataframe
df_delimitado = df[["titulo", "data"]]
df_delimitado

,titulo,data
1,Abertura de mercado para o Brasil no Chile - N...,29/04/2026
2,Promulgação do Acordo Provisório de Comércio e...,28/04/2026
3,Mortes de brasileiros no Líbano em decorrência...,27/04/2026
4,Ataques terroristas no Mali,27/04/2026
5,Atentado na Colômbia,27/04/2026
6,Apresentação da candidatura do Professor Georg...,24/04/2026
7,"Aberturas de mercado para o Brasil em Cuba, na...",24/04/2026
8,Aberturas de mercado para o Brasil no Togo - N...,22/04/2026
9,Atos adotados por ocasião da Visita de Estado ...,20/04/2026
10,Declaração Conjunta Brasil-Alemanha sobre apoi...,20/04/2026


In [36]:

# primeiras (head), ultimas (tail) e linhas aleatórias (sample)
df.head(10)

,titulo,link,data,hora,num_nota,paragrafo
1,Abertura de mercado para o Brasil no Chile - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,29/04/2026,17h49,150,[O governo brasileiro concluiu negociações que...
2,Promulgação do Acordo Provisório de Comércio e...,https://www.gov.br/mre/pt-br/canais_atendiment...,28/04/2026,19h20,149,[O Presidente Luiz Inácio Lula da Silva assino...
3,Mortes de brasileiros no Líbano em decorrência...,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,19h04,148,"[O governo brasileiro tomou conhecimento, com ..."
4,Ataques terroristas no Mali,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,13h38,147,[O Governo brasileiro manifesta grave preocupa...
5,Atentado na Colômbia,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,08h45,146,[O governo brasileiro condena o ataque perpetr...
6,Apresentação da candidatura do Professor Georg...,https://www.gov.br/mre/pt-br/canais_atendiment...,24/04/2026,15h57,145,"[Realizou-se, em 23 de abril, coquetel de apre..."
7,"Aberturas de mercado para o Brasil em Cuba, na...",https://www.gov.br/mre/pt-br/canais_atendiment...,24/04/2026,15h48,144,[O governo brasileiro concluiu negociações que...
8,Aberturas de mercado para o Brasil no Togo - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,22/04/2026,19h20,143,[O governo brasileiro concluiu negociações que...
9,Atos adotados por ocasião da Visita de Estado ...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,13h13,142,[Foram adotados os seguintes atos por ocasião ...
10,Declaração Conjunta Brasil-Alemanha sobre apoi...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,11h12,141,[No contexto da reunião bilateral entre o Pres...


In [37]:
df.describe(include="all")

,titulo,link,data,hora,num_nota,paragrafo
count,30,30,30,30,30,30
unique,30,30,15,28,30,30
top,Abertura de mercado para o Brasil no Chile - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,19h20,150,[O governo brasileiro concluiu negociações que...
freq,1,1,4,2,1,1


In [38]:
df.isnull().sum()

,0
titulo,0
link,0
data,0
hora,0
num_nota,0
paragrafo,0


In [39]:
# verificar linhas duplicadas
df.duplicated().sum()

TypeError: unhashable type: 'list'